# PyTorch Foundations 04 - Deliver the rows. Pack the right state.

Part 03's fuel model works, but every update reads all 96 rows. What happens
when examples arrive in smaller groups—or the Python process disappears?
Those are data-delivery and saved-state problems, not new prediction math.

**Guiding question:** What must stay together when you shuffle, batch, or save?

| Complaint | Evidence we will collect |
|---|---|
| A full-table update hides grouping | Manually divide shuffled row addresses |
| Manual slicing spreads through training code | Let Dataset pair rows and DataLoader deliver them |
| A file exists, but can another object use it? | Load the same weights into fresh models and compare predictions |
| Matching predictions isn't matching training | Inspect what the training snapshot actually saves |
| A rate looks fast in one lucky run | Reset the model and change only learning rate |

![Ninety-six paired fuel examples are shuffled by row and delivered in batches of twenty, twenty, twenty, twenty and sixteen.](../images/04-deliver-every-row.png)

Keep **engine load**, **temperature**, and the synthetic fuel target. The
setup finds Part 03's `part03_linear_state.pt` and `part03_config.json`.
If either is missing, run Part 03 first; this notebook won't make replacement
weights and pretend the handoff worked.

We will measure delivery and prediction reload, then write a training-state
snapshot. We won't restore an interrupted run and compare its next update.
[Handwritten theory](04-data-checkpoints-and-hyperparameters-theory.md).


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")
from pathlib import Path
LESSON_DIR = next(
    folder for folder in (
        Path(globals().get("__vsc_ipynb_file__", "")).resolve().parent,
        Path.cwd(),
        Path.cwd().joinpath('learning', '02-pytorch-foundations', '04-data-checkpoints-and-hyperparameters'),
    )
    if (folder / "04-data-checkpoints-and-hyperparameters.ipynb").is_file()
)

# -- Imports, model definition, and artifact paths ----------------------------
import json
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FuelLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, inputs):
        # (B sensor rows, 2 features) -> (B, 1 fuel target): keep one output per row.
        return self.linear(inputs)

part03_dir = LESSON_DIR.parent / "03-models-and-training-loops" / "artifacts"
state_path = part03_dir / "part03_linear_state.pt"
config_path = part03_dir / "part03_config.json"
if not state_path.exists() or not config_path.exists():
    raise FileNotFoundError("Part 03 artifacts are missing. Execute Part 03 before Part 04.")
print("Part 03 artifacts found; fresh-kernel reload can proceed.")


### Build the case before putting the contents back

The saved keys describe a two-input, one-output linear layer. The class in
setup supplies that structure; `load_state_dict` assigns the saved weight
and bias to it. `map_location` chooses where those tensors land.
Only load checkpoint files you trust.

Read the JSON as well: two input features and one output match this class.
The feature meanings still come from us—load first, temperature second.
`eval()` chooses prediction behavior; it neither loads nor trains weights.

The next cell checks dimensions, loads state, and prints its keys. A
successful assignment proves compatibility. We'll test matching answers
after rebuilding the input rows too.


In [ ]:
# -- Reload inter-notebook state and verify reconstruction facts ---------------
config = json.loads(config_path.read_text(encoding="utf-8"))
assert config["input_features"] == 2 and config["output_features"] == 1
reloaded_model = FuelLinearModel().to(device)
reloaded_model.load_state_dict(torch.load(state_path, map_location=device, weights_only=True))
reloaded_model.eval()
print(f"Reloaded keys: {list(reloaded_model.state_dict())}")
print(f"Configuration: {config}")


## 1 - Recreate the pairs before changing delivery

Each row is still `[engine load, ambient temperature]`, checked against
one fuel value. The saved seed and row count let this synthetic lesson
rebuild its input table and noise recipe. Real data also needs a preserved
source and preprocessing history; a seed cannot recreate observations.

The next cell stacks two lists into `[96,2]` inputs and keeps targets
`[96,1]`. It generates the noise on CPU here; Part 03 generated it on the
selected device. So cross-device runs should not assume bit-identical noisy
targets. Our later reload test compares predictions on the **same probe
inputs**, not cross-device equality of the original training losses.

With pairs in place, the next problem is delivery: one epoch should not
quietly lose or duplicate rows.


In [ ]:
# -- Recreate the deterministic data contract from Part 03 -------------------
torch.manual_seed(config["seed"])
n_examples = config["n_examples"]
engine_load = torch.linspace(0.1, 1.0, n_examples)
ambient_temp = torch.linspace(-0.8, 0.9, n_examples).roll(17)
# Two (96 rows,) vectors -> (96, 2): restore the same feature-column order
# used by the saved model, or its learned weights would read the wrong sensor.
X = torch.stack([engine_load, ambient_temp], dim=1)
noise = 0.03 * torch.randn(n_examples, 1)
y = 1.8 * X[:, :1] - 0.65 * X[:, 1:2] + 0.4 + noise
print(f"Recreated X={tuple(X.shape)}, y={tuple(y.shape)} from saved configuration.")


### Shuffle envelopes, not their contents

Put each input and its fuel answer in one envelope. Shuffle envelope
**addresses**, then take twenty at a time:

```text
96 addresses -> 20 + 20 + 20 + 20 + 16
```

The last group is smaller, not defective. Dropping it would train on only
80 rows this epoch. A total of 96 isn't enough either: a duplicate could
hide an omission. Joining and sorting the addresses lets the next cell
check the entire original address list.

> **Visual to add — The last sixteen envelopes still count**
> Show 96 paired fuel envelopes moving into four boxes labelled 20 and
> one labelled 16. Keep load/temperature and the target sealed together.
> Beside it show a crossed-out dropped remainder with "80 rows delivered",
> versus the retained remainder with "96, each once". Don't use a generic
> pipeline of named boxes. Suggested caption: "Shuffle changes who arrives
> together, not which answer belongs to a reading."

**Predict:** Will shuffled delivery still cover every original row once?
The equality assertion tests identities, not just batch sizes.


In [ ]:
# -- Build shuffled manual batches and expose the remainder policy -------------
batch_size = 20
generator = torch.Generator().manual_seed(31)
permutation = torch.randperm(n_examples, generator=generator)
manual_batches = [
    permutation[start : start + batch_size]
    for start in range(0, n_examples, batch_size)
]
# Index groups (20,), (20,), (20,), (20,), (16,) -> (96,): join row IDs
# to check every example appears once; sorting removes delivery order for this check.
covered = torch.cat(manual_batches).sort().values
print(f"Batch sizes: {[len(indices) for indices in manual_batches]}")
print(f"All examples covered once: {torch.equal(covered, torch.arange(n_examples))}")
print("The smaller final batch is retained deliberately.")
assert torch.equal(covered, torch.arange(n_examples))


### Repeating those slices everywhere is the real nuisance

`TensorDataset(X,y)` keeps row partners together. Ask it for one item:
two sensor values `[2]` and one target `[1]`.
`DataLoader` decides order and grouping: a full delivery has inputs
`[20,2]` and targets `[20,1]`.

The loader doesn't normalize sensors, compute a loss, or update weights.
Its own random generator makes shuffling repeatable without relying on
model-initialization draws.

Taking `next(iter(loader))` is a **real iteration**, not a frozen preview:
it advances generator state. Don't expect the next iterator to replay that
same first delivery. The next cell prints one item and one batch so the
extra example axis is visible.


In [ ]:
# -- Express example and batch contracts with Dataset and DataLoader ----------
# Pair features (96, 2) with targets (96, 1); the loader groups these
# into (B, 2) and (B, 1) without changing which target belongs to each row.
dataset = TensorDataset(X, y)
loader_generator = torch.Generator().manual_seed(31)
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=loader_generator,
    drop_last=False,
)
first_X, first_y = next(iter(loader))
print(f"One example: X{tuple(dataset[0][0].shape)}, y{tuple(dataset[0][1].shape)}")
print(f"One batch: X{tuple(first_X.shape)}, y{tuple(first_y.shape)}")


### Audit delivery without the shuffle

An ordered loader should rebuild the original table when its batches are
joined along the row axis. `cat` extends an axis that already exists;
`stack` would add a new batch-number axis, and the 16-row remainder
wouldn't even match the full groups.

The next cell joins inputs and targets separately, checks exact equality
with `X` and `y`, and checks the final batch length of 16. The earlier
manual audit checked shuffled **addresses**; this one checks ordered
**contents**. A first batch that looks right proves neither by itself.


In [ ]:
# -- Prove the DataLoader covers the same examples without dropping -----------
ordered_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
# Five (B, 2) feature batches -> (96, 2): join rows, not feature columns,
# to verify the ordered loader rebuilds the original table.
loaded_X = torch.cat([batch_X for batch_X, _ in ordered_loader])
# Five (B, 1) target batches -> (96, 1): keep targets aligned with those rows.
loaded_y = torch.cat([batch_y for _, batch_y in ordered_loader])
print(f"Feature parity: {torch.equal(loaded_X, X)}")
print(f"Target parity: {torch.equal(loaded_y, y)}")
print(f"Last batch size: {len(list(ordered_loader)[-1][0])}")
assert torch.equal(loaded_X, X) and torch.equal(loaded_y, y)
assert len(list(ordered_loader)[-1][0]) == 16


We can now deliver the pairs without losing them. Saving asks a different
question: do you want the same **answers**, or the same **next learning
step**? Adam's memory and the next shuffled batch matter for the second,
even if they don't appear in the model's forward formula.


## 2 - Same predictions is a useful, smaller promise

![An inference case can reproduce predictions; the Part 04 training bag saves model, Adam, epoch and configuration but omits loader random state and is not recovery-tested.](../images/04-snapshot-is-not-recovery.png)

The opening reconstructed one model. Now build a second independent one
and load the same saved weights. Give both the same first seven input rows.
Inference mode avoids recording graphs; eval mode selects layer behavior.
No update occurs between the two predictions.

The largest absolute prediction gap should be zero, and the next cell
asserts exactly that. This checks two reconstructions of the saved function.
It doesn't compare against a live Part 03 object, and it doesn't restore
an optimizer or a next batch.

> **Visual to add — Two identical guessers, different updater memories**
> Give two rebuilt fuel models the same seven-row probe and matching
> weight/bias cards. Draw matching prediction columns, without invented
> values. Under them, show Adam on one side with accumulated gradient
> summaries and a fresh Adam on the other with empty memory; place a
> shuffled next-batch deck beside each. Suggested caption: "Identical
> guesses don't tell you which update comes next."


In [ ]:
# -- Prove save/load prediction parity ----------------------------------------
probe = X[:7].to(device)
with torch.inference_mode():
    before_reload = reloaded_model(probe)
parity_model = FuelLinearModel().to(device)
parity_model.load_state_dict(torch.load(state_path, map_location=device, weights_only=True))
parity_model.eval()
with torch.inference_mode():
    after_reload = parity_model(probe)
# Probe differences (7, 1) -> scalar maximum: inspect the worst prediction gap.
max_reload_gap = (before_reload - after_reload).abs().max().item()
print(f"Maximum reload prediction gap: {max_reload_gap:.2e}")
assert max_reload_gap == 0.0


### Now save more than the prediction machine

The next block trains a **fresh** model, not the reloaded Part 03 model.
Each batch moves to the selected device, clears feedback, predicts, scores,
backpropagates, and takes one Adam step.

At batch size 20, 96 rows require five updates per epoch. Twenty-five
epochs therefore make **125 updates**. The final saved `epoch` is 24:
Python counts from zero, while the message reports 25 completed passes.

The snapshot packs model weights, Adam's state, progress, and selected
settings. Saving captures their current contents; it doesn't freeze all
random generators or prove a future continuation. We'll name exactly
what is missing beside the file, once rather than after every line.


In [ ]:
# -- Train mini-batches and save a training-state snapshot ------------------------------
train_model = FuelLinearModel().to(device)
optimizer = torch.optim.Adam(train_model.parameters(), lr=0.03)
loss_function = nn.MSELoss()
loader_generator = torch.Generator().manual_seed(31)
train_loader = DataLoader(dataset, batch_size=20, shuffle=True, generator=loader_generator)
for epoch in range(25):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = loss_function(train_model(batch_X), batch_y)
        loss.backward()
        optimizer.step()

artifact_dir = LESSON_DIR / "artifacts"
artifact_dir.mkdir(exist_ok=True)
resume_path = artifact_dir / "part04_resumable.pt"
torch.save({
    "epoch": epoch,
    "model_state": train_model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "config": {"batch_size": 20, "learning_rate": 0.03},
}, resume_path)
print(f"Saved training-state snapshot at epoch {epoch + 1}: {resume_path}")


### Read the saved bundle by job

| Saved field | What it contributes |
|---|---|
| `model_state` | The fuel prediction's learned tensors |
| `optimizer_state` | Adam's running update summaries and counters |
| `epoch` | Zero-based index of the last completed pass |
| `config` | Batch size 20 and learning rate 0.03 |

The historical file is named `part04_resumable.pt`, but this lesson only
**writes** it. It leaves out the shuffled loader generator state and never
tests the same next update after restoration. To test recovery, restore
everything affecting that next batch and step, then compare with an
uninterrupted run. Inference alone doesn't need Adam's memory.


## 3 - A faster-looking run needs a fair comparison

A larger rate can move farther, overshoot, or diverge. The next helper
resets seed 41 and creates a fresh model for each rate:
`0.005`, `0.05`, and `0.5`.

It keeps inputs, targets, SGD, MSE, and **30 full-batch updates** fixed.
It returns the list of pre-update losses, not a validation score or a
selected production setting. The final printed entry was measured just
before the last update.

**Predict:** Which rate will make most progress within this budget?
Read the measured endpoints; no ranking is promised. Changing batch size
at the same time would also change updates per epoch and muddy the answer.


In [ ]:
# -- Compare one variable: learning rate --------------------------------------
def loss_after_updates(learning_rate, updates=30):
    torch.manual_seed(41)
    candidate = FuelLinearModel().to(device)
    candidate_optimizer = torch.optim.SGD(candidate.parameters(), lr=learning_rate)
    local_X, local_y = X.to(device), y.to(device)
    losses = []
    for _ in range(updates):
        candidate_optimizer.zero_grad()
        candidate_loss = loss_function(candidate(local_X), local_y)
        candidate_loss.backward()
        candidate_optimizer.step()
        losses.append(candidate_loss.item())
    return losses

for learning_rate in (0.005, 0.05, 0.5):
    losses = loss_after_updates(learning_rate)
    print(f"lr={learning_rate:5.3f}: {losses[0]:.4f} -> {losses[-1]:.4f}")


### Your turn - count the groups without retraining

Set `YOUR_BATCH_SIZE=16`: 96 rows should arrive as **six full groups**.
At 20, expect five groups with a 16-row remainder. Change only that value
and compare the printed sizes.

The sum check catches missing row counts; the ordered reconstruction above
was the stronger check of actual contents. This drill is about delivery,
not claiming that different update counts make a fair training comparison.


In [ ]:
# -- CHANGE THIS: inspect the resulting batch contract ------------------------
YOUR_BATCH_SIZE = 16
your_loader = DataLoader(dataset, batch_size=YOUR_BATCH_SIZE, shuffle=False, drop_last=False)
your_sizes = [len(batch_X) for batch_X, _ in your_loader]
print(f"Batch sizes at {YOUR_BATCH_SIZE}: {your_sizes}")
assert sum(your_sizes) == len(dataset)


### One more batching problem is waiting in the language route

Our fuel rows all have two measurements. Text examples can have different
numbers of positions. To put them in one rectangle, add **padding** and
retain true **lengths**. The blanks aren't observations.

A **mask** says which positions count for a particular job: which positions
the model may read, or which targets the loss should grade. Those are
separate decisions; skipping a padding target in loss doesn't by itself
stop a model reading padded inputs. Stage 03 builds that machinery.


## The short version

- Pair first, shuffle addresses second. Twenty-at-a-time delivery of
  96 examples means four full batches plus sixteen, not four batches.
- A Dataset retrieves a pair. A DataLoader chooses how pairs arrive.
  Neither chooses your loss or trains the model.
- The manual check verifies every row address once; ordered concatenation
  verifies the delivered inputs and targets themselves.
- Freshly loaded weights can reproduce predictions. The seven-row check
  says nothing about a next optimizer step.
- Adam remembers update history outside the model. This snapshot writes
  that memory but omits loader random state and is not recovery-tested.
- Compare learning rates from the same start and update budget. Use
  validation, not these training losses, when choosing a setting to keep.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Manual batch identities; item/batch shapes; ordered-loader contents; saved-weight prediction equality; mini-batch training; writing selected training state; rate and batch-size comparisons |
| Explained | Model versus optimizer/progress/configuration state; next-update recovery; padding/length/mask roles |
| Later | Validation, modes, and model selection in Part 05; sequence batching in Stage 03; worker tuning, distributed sampling, and recovery testing aren't implemented here |

**Next:** [The Modern Training Lifecycle](../05-modern-training-lifecycle/05-modern-training-lifecycle.ipynb)
keeps the fuel task but separates learning from checking, chooses which
model to keep, and saves the preprocessing needed to use it.
